In [9]:
import pandas as pd
import numpy as np

import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/processed/credit_risk_cleaned.csv")
print(f"Dataset shape: {df.shape}")

# Ordinal encoding for loan grade
grade_mapping = {
    'A': 1,
    'B': 2,
    'C': 3,
    'D': 4,
    'E': 5,
    'F': 6
}

df['loan_grade'] = df['loan_grade'].map(grade_mapping)

# Home ownership encoding
home_mapping = {
    'OWN': 0,
    'RENT': 1,
    'MORTGAGE': 2,
    'OTHER': 3
}

df['person_home_ownership'] = (
    df['person_home_ownership']
    .map(home_mapping)
)

# Previous default encoding
default_mapping = {
    'N': 0,
    'Y': 1
}

df['cb_person_default_on_file'] = (
    df['cb_person_default_on_file']
    .map(default_mapping)
)

# =====================================================
# ONE-HOT ENCODE REMAINING NOMINAL FEATURE
# =====================================================

df_encoded = pd.get_dummies(
    df,
    columns=['loan_intent'],
    drop_first=True
)


# Convert boolean dummies to integer
dummy_cols = df_encoded.select_dtypes(
    include=['bool']
).columns


df_encoded[dummy_cols] = (
    df_encoded[dummy_cols]
    .astype(int)
)

output_file = "data/processed/loan_data_preprocessed.csv"

df_encoded.to_csv(output_file, index=False)

print(f"Preprocessed dataset saved successfully as: {output_file}")
print(f"Dataset shape: {df_encoded.shape}")

Dataset shape: (29567, 12)
Preprocessed dataset saved successfully as: data/processed/loan_data_preprocessed.csv
Dataset shape: (29567, 16)


In [1]:
# =====================================================
# LOAN RISK PREDICTION USING OLS & LOGISTIC REGRESSION
# =====================================================

import pandas as pd
import numpy as np

import statsmodels.api as sm
import statsmodels.formula.api as smf

from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan

from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("data/processed/loan_data_preprocessed.csv")

print(df.head())

# =====================================================
# DATA PREPROCESSING
# =====================================================

categorical_cols = [
    'person_home_ownership',
    'loan_intent',
    'loan_grade',
    'cb_person_default_on_file'
]

df_encoded = pd.get_dummies(
    df,
    columns=categorical_cols,
    drop_first=True
)

# =====================================================
# DEFINE FEATURES AND TARGET
# =====================================================

X = df_encoded.drop("loan_status", axis=1)
y = df_encoded["loan_status"]

X = sm.add_constant(X)

# =====================================================
# 1. MULTIPLE LINEAR REGRESSION (OLS)
# =====================================================

ols_model = sm.OLS(y, X).fit()

print("\n================ OLS SUMMARY ================\n")
print(ols_model.summary())

# =====================================================
# 2. VIF CHECK
# =====================================================

vif_data = pd.DataFrame()

vif_data["Feature"] = X.columns

vif_data["VIF"] = [
    variance_inflation_factor(X.values, i)
    for i in range(X.shape[1])
]

print("\n================ VIF =================\n")
print(vif_data.sort_values("VIF", ascending=False))

# =====================================================
# 3. QQ PLOT OF RESIDUALS
# =====================================================

residuals = ols_model.resid

plt.figure(figsize=(8,6))
stats.probplot(
    residuals,
    dist="norm",
    plot=plt
)

plt.title("Q-Q Plot of OLS Residuals")
plt.show()

# =====================================================
# 4. BREUSCH-PAGAN TEST
# =====================================================

bp_test = het_breuschpagan(
    residuals,
    ols_model.model.exog
)

bp_labels = [
    "Lagrange Multiplier Statistic",
    "p-value",
    "f-value",
    "f p-value"
]

print("\n=========== BREUSCH PAGAN TEST ===========")

for label, value in zip(bp_labels, bp_test):
    print(f"{label}: {value}")

# =====================================================
# 5. LOGISTIC REGRESSION (GLM)
# =====================================================

glm_model = sm.GLM(
    y,
    X,
    family=sm.families.Binomial()
).fit()

print("\n================ GLM SUMMARY ================\n")
print(glm_model.summary())

# =====================================================
# PREDICTION PROBABILITIES
# =====================================================

pred_prob = glm_model.predict(X)

df["default_probability"] = pred_prob

print("\nTop Predicted Probabilities")
print(
    df[
        ["loan_status", "default_probability"]
    ].head()
)

# =====================================================
# ODDS RATIOS
# =====================================================

odds_ratios = pd.DataFrame({
    "Variable": glm_model.params.index,
    "Odds Ratio": np.exp(glm_model.params.values)
})

print("\n================ ODDS RATIOS ================\n")
print(odds_ratios.sort_values(
    "Odds Ratio",
    ascending=False
))

   person_age  person_income person_home_ownership  person_emp_length  \
0          21           9600                   OWN                5.0   
1          25           9600              MORTGAGE                1.0   
2          23          65500                  RENT                4.0   
3          24          54400                  RENT                8.0   
4          21           9900                   OWN                2.0   

  loan_intent loan_grade  loan_amnt  loan_int_rate  loan_status  \
0   EDUCATION          B       1000          11.14            0   
1     MEDICAL          C       5500          12.87            1   
2     MEDICAL          C      35000          15.23            1   
3     MEDICAL          C      35000          14.27            1   
4     VENTURE          A       2500           7.14            1   

   loan_percent_income cb_person_default_on_file  cb_person_cred_hist_length  
0                 0.10                         N                           2  


ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).